# Example 12.1 / 2: Set Targets with Goal Programming

Companion notebook for **12.1 Multiple Objectives** in the AMPL lab Documentation, section "Set Targets with Goal Programming". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Set Targets with Goal Programming

When the policy is expressed as target levels rather than exchange rates, introduce nonnegative
deviation variables for underachievement and overachievement. For a target $g_k$, write

$$
f_k(x)+d_k^- - d_k^+ = g_k,
\qquad d_k^-,d_k^+\ge0,
$$

and minimize only the deviations that represent undesirable outcomes. Weighted goal programming
puts selected deviations in one scaled objective; preemptive goal programming assigns priority
levels and solves them lexicographically. State whether missing a target and exceeding it have
different costs, because penalizing both sides can encode a different policy.

Goal programming is closely related to Soft Constraints and Penalties (AMPL lab documentation, section 10.5), but a target
deviation represents an explicit performance goal rather than silently weakening a hard physical
or safety requirement. Report the achieved value and both deviations for every goal.

On the two-plant instance, set a cost target of 28 and an emission target of 25. Only exceeding a
target is undesirable, so the weighted objective penalizes `OverCost` and `OverEmission` only.
The run then solves the preemptive version, cost goal first, by dropping the weighted objective
with `drop()` and giving the two deviation objectives priorities through the `.objpriority`
suffix, which [Let the Solver Run the Passes](#let-the-solver-run-the-passes) explains:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PLANTS;                           # production plants
param unit_cost {PLANTS} >= 0;        # dollars per unit
param unit_emission {PLANTS} >= 0;    # kg CO2 per unit
param capacity {PLANTS} >= 0;         # units
param demand >= 0;                    # units
param cost_target;                    # dollars
param emission_target;                # kg CO2

var Make {p in PLANTS} >= 0, <= capacity[p];  # units produced at each plant
var UnderCost >= 0;                   # cost below target, dollars
var OverCost >= 0;                    # cost above target, dollars
var UnderEmission >= 0;               # emissions below target, kg CO2
var OverEmission >= 0;                # emissions above target, kg CO2

minimize WeightedDeviation: OverCost + OverEmission;  # weighted goal programming: total overrun
minimize CostOverrun: OverCost;       # preemptive goal 1 (cost-first policy)
minimize EmissionOverrun: OverEmission;  # preemptive goal 2 (cost-first policy)

subject to MeetDemand: sum {p in PLANTS} Make[p] = demand;  # total output covers demand exactly
subject to CostGoal:                  # cost plus deviations equals the cost target
    sum {p in PLANTS} unit_cost[p] * Make[p] + UnderCost - OverCost = cost_target;
subject to EmissionGoal:              # emissions plus deviations equal the emission target
    sum {p in PLANTS} unit_emission[p] * Make[p] + UnderEmission - OverEmission
        = emission_target;

suffix objpriority IN, integer;       # solve order for the preemptive goals

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PLANTS"] = ["A", "B"]
ampl.param["unit_cost"] = {"A": 2, "B": 4}
ampl.param["unit_emission"] = {"A": 5, "B": 1}
ampl.param["capacity"] = {"A": 8, "B": 8}
ampl.param["demand"] = 10
ampl.param["cost_target"] = 28
ampl.param["emission_target"] = 25

In [5]:
# run.py
def report(label):
    values = {name: ampl.var[name].value()
              for name in ["UnderCost", "OverCost", "UnderEmission", "OverEmission"]}
    print(label, ampl.solve_result, ampl.var["Make"].to_dict(), values)

ampl.solve(solver="highs", verbose=False)
report("weighted")

ampl.obj["WeightedDeviation"].drop()
ampl.eval("let CostOverrun.objpriority := 2; let EmissionOverrun.objpriority := 1;")
ampl.solve(solver="highs", highs_options="obj:multi=2", verbose=False)
report("cost first")

weighted solved {'A': 3.75, 'B': 6.25} {'UnderCost': 0.0, 'OverCost': 4.5, 'UnderEmission': 0.0, 'OverEmission': 0.0}
cost first solved {'A': 6, 'B': 4} {'UnderCost': 0.0, 'OverCost': 0.0, 'UnderEmission': 0.0, 'OverEmission': 9.0}


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PLANTS := A B;
param: unit_cost unit_emission capacity :=
    A    2   5   8
    B    4   1   8 ;
param demand := 10;
param cost_target := 28;
param emission_target := 25;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs, solver_msg 0;
solve;
display Make, UnderCost, OverCost, UnderEmission, OverEmission;
drop WeightedDeviation;
let CostOverrun.objpriority := 2;
let EmissionOverrun.objpriority := 1;
option highs_options 'obj:multi=2';
solve;
display Make, UnderCost, OverCost, UnderEmission, OverEmission;

Make [*] :=
A  3.75
B  6.25
;

UnderCost = 0
OverCost = 4.5
UnderEmission = 0
OverEmission = 0

HiGHS 1.14.0:   obj:multi = 2
Make [*] :=
A  6
B  4
;

UnderCost = 0
OverCost = 0
UnderEmission = 0
OverEmission = 9

